In [3]:
import torch 
import torch.nn as nn 
import torch.nn.functional as F
import pandas as pd 
import numpy as np
import time 
from tqdm import tqdm

In [4]:
df = pd.read_pickle('data/New folder/train.pkl')
df

,series_id,time_step,close,volume
0,1,0,0.13700,171985.703125
1,1,1,0.13656,85451.398438
2,1,2,0.13647,121151.898438
3,1,3,0.13693,249110.593750
4,1,4,0.13715,280344.500000
...,...,...,...,...
18331219,50,394555,0.37340,54432.101562
18331220,50,394556,0.37360,59362.000000
18331221,50,394557,0.37360,21189.900391
18331222,50,394558,0.37360,36969.500000


In [5]:
# -----------------------------
# 1) Simple LSTM Model
# -----------------------------
class LSTMModel(nn.Module):
    def __init__(self, input_size=2, hidden_size=64, num_layers=2, output_size=10, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x, hc=None):
        out, (h,c) = self.lstm(x, hc)          # (B, seq_len, hidden)
        out = self.fc(out[:, -1, :])   # use last hidden state
        return out, (h,c)

In [6]:
model = LSTMModel()

print(model)

LSTMModel(
  (lstm): LSTM(2, 64, num_layers=2, batch_first=True, dropout=0.2)
  (fc): Linear(in_features=64, out_features=10, bias=True)
)


IGNORE FROM HERE 
------------------------------

In [4]:
# --------------------------
# 1) Load & sort data
# --------------------------
from tqdm import tqdm
import time

TRAIN_PATH = "train.pkl"
print("reading training data...\n.") 

t0 = time.perf_counter()
train_df = pd.read_pickle(TRAIN_PATH)
train_df = train_df.sort_values(["series_id", "time_step"]).reset_index(drop=True)
t_load = time.perf_counter() - t0
print(f"loaded training data in {t_load:.3f}s")

# Expect columns: series_id, time_step, close, volume


reading training data...
.
loaded training data in 2.115s


In [7]:
new_train_df = pd.DataFrame()

new = train_df.groupby("series_id").agg(mean_close=("close", "mean"),
                                        std_close=("close", "std"),
                                        mean_volume=("volume", "mean"),
                                        std_volume=("volume", "std")).reset_index()

new

,series_id,mean_close,std_close,mean_volume,std_volume
0,1,0.100583,0.029522,6.413277e+04,1.489433e+05
1,2,0.533405,0.148294,8.134202e+03,2.696395e+04
2,3,15.354797,5.510595,1.118376e+03,2.894890e+03
3,4,9173.823242,1569.836182,4.810945e+01,8.013607e+01
4,5,0.746253,0.292024,1.340342e+05,2.176044e+05
5,6,5.217080,1.582593,1.102515e+04,1.922328e+04
6,7,16.192724,7.643600,1.513665e+03,3.952120e+03
7,8,0.348513,0.055784,1.379092e+04,4.359222e+04
8,9,0.081676,0.021421,3.407534e+04,1.126295e+05
9,10,0.790860,0.210102,1.180911e+04,4.353254e+04


In [8]:
new = new[~new['series_id'].isin([4,23,32,33,42])]

new

,series_id,mean_close,std_close,mean_volume,std_volume
0,1,0.100583,0.029522,6.413277e+04,1.489433e+05
1,2,0.533405,0.148294,8.134202e+03,2.696395e+04
2,3,15.354797,5.510595,1.118376e+03,2.894890e+03
4,5,0.746253,0.292024,1.340342e+05,2.176044e+05
5,6,5.217080,1.582593,1.102515e+04,1.922328e+04
6,7,16.192724,7.643600,1.513665e+03,3.952120e+03
7,8,0.348513,0.055784,1.379092e+04,4.359222e+04
8,9,0.081676,0.021421,3.407534e+04,1.126295e+05
9,10,0.790860,0.210102,1.180911e+04,4.353254e+04
10,11,0.052794,0.017299,3.348659e+04,2.000074e+05


In [9]:
train_df = train_df[~train_df['series_id'].isin([4,23,32,33,42])]

train_df

,series_id,time_step,close,volume
0,1,0,0.13700,171985.703125
1,1,1,0.13656,85451.398438
2,1,2,0.13647,121151.898438
3,1,3,0.13693,249110.593750
4,1,4,0.13715,280344.500000
...,...,...,...,...
18331219,50,394555,0.37340,54432.101562
18331220,50,394556,0.37360,59362.000000
18331221,50,394557,0.37360,21189.900391
18331222,50,394558,0.37360,36969.500000


In [10]:
train_df['volume'].iloc[0:10] 

0    171985.703125
1     85451.398438
2    121151.898438
3    249110.593750
4    280344.500000
5    291702.093750
6    108030.000000
7    149102.906250
8     31606.500000
9    168079.593750
Name: volume, dtype: float32

In [13]:
np.log1p(train_df['volume'].iloc[0:10])

0    12.055173
1    11.355715
2    11.704808
3    12.425656
4    12.543778
5    12.583491
6    11.590174
7    11.912398
8    10.361150
9    12.032199
Name: volume, dtype: float32

START FROM HERE 
---------------------

In [ ]:
import pandas as pd

# --------------------------------------------------
# 1) Load the raw training data
# --------------------------------------------------
df = pd.read_pickle("data/New folder/train.pkl")   # original dataset

# --------------------------------------------------
# 2) Drop inconsistent / high-variance series
# --------------------------------------------------
bad_series = [4, 23, 32, 33, 42]
df = df[~df['series_id'].isin(bad_series)].reset_index(drop=True)

# --------------------------------------------------
# 3) Sort by series_id and time_step for consistency
# --------------------------------------------------
df = df.sort_values(['series_id', 'time_step']).reset_index(drop=True)

# --------------------------------------------------
# 4) Split by series_id (80% train / 20% validation)
# --------------------------------------------------
unique_series = sorted(df['series_id'].unique())
n_train = int(0.8 * len(unique_series))

train_ids = unique_series[:n_train]
val_ids   = unique_series[n_train:]

train_df = df[df['series_id'].isin(train_ids)].reset_index(drop=True)
val_df   = df[df['series_id'].isin(val_ids)].reset_index(drop=True)

# --------------------------------------------------
# 5) Save the processed data
# --------------------------------------------------
train_df.to_pickle("data/train_split_train.pkl")
val_df.to_pickle("data/train_split_val.pkl")


In [8]:
from torch.utils.data import IterableDataset
import torch
import numpy as np
from dataset import TrainWindowSampler

class WindowsDataset(IterableDataset):
    """
    Stream overlapping time-series windows per series_id.
    - stride = 5 (overlapping windows)
    - close: kept as-is
    - volume: log1p transformed only
    - no min–max or any other normalization

    Each sample:
        X: (60, 2) float32 -> [close_ratio, log1p(volume)]
        y: (10,)  float32  -> next 10 close ratios
    """

    def __init__(self, train_path: str, step_size: int = 5, seed: int = 1337):
        self.train_path = train_path
        self.step_size = step_size
        self.seed = seed

    def __iter__(self):
        sampler = TrainWindowSampler(
            train_path=self.train_path,
            window=70,
            input_len=60,
            horizon_len=10,
            rolling=True,
            step_size=self.step_size,
            seed=self.seed,
        )

        for X, y in sampler.iter_windows():
            # X: (60,2) -> [close, volume]
            # y: (10,)  -> next 10 closes

            close = X[:, 0].astype(np.float32)                # leave unchanged
            vol   = np.log1p(np.clip(X[:, 1], 0, None)).astype(np.float32)

            X_out = np.stack([close, vol], axis=1)
            y_out = y.astype(np.float32)

            yield torch.from_numpy(X_out), torch.from_numpy(y_out)


In [9]:
from torch.utils.data import DataLoader

train_ds = WindowsDataset(
    train_path="data/train_split_train.pkl", 
    step_size=5,
    seed=1337
)

val_ds = WindowsDataset(
    train_path="data/train_split_val.pkl",    
    step_size=5,
    seed=1337
)

train_loader = DataLoader(train_ds, batch_size=256, num_workers=0)
val_loader   = DataLoader(val_ds, batch_size=256, num_workers=0)


In [10]:
device = "cuda"

In [11]:
# --------------------------
# 4) Model, loss, optim
# --------------------------
print("building model...\n.")
t0 = time.perf_counter()
model = LSTMModel().to(device)
opt = torch.optim.AdamW(model.parameters(), lr=0.0001, weight_decay=1e-4)
crit = nn.MSELoss()
t_model_build = time.perf_counter() - t0
print(f"model built in {t_model_build:.3f}s")


building model...
.
model built in 1.281s


In [12]:
EPOCHS = 10
model.train()
for epoch in range(1, EPOCHS + 1):
    total_loss = 0.0
    total_samples = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}", ncols=100)
    for xb, yb in pbar:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        loss = crit(model(xb), yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        opt.step()
        total_loss += loss.item() * xb.size(0)
        total_samples += xb.size(0)
        pbar.set_postfix(loss=f"{total_loss / total_samples:.6f}")
    print(f"Epoch {epoch}: train_mse={total_loss / total_samples:.6f}")


Epoch 1/10: 0it [00:01, ?it/s]


AttributeError: 'tuple' object has no attribute 'size'